# End-to-End Provider Directory Update Pipeline Demo

This notebook demonstrates the full AI-powered pipeline on sample healthcare provider data.

## Step 1: Setup & Initialize Database

In [1]:
import sys
import os
sys.path.insert(0, "..")
os.chdir("..")

from pipeline.db.store import get_db, init_db
from pipeline.db.seed import seed_db
from pipeline.orchestrator import run_pipeline, run_batch
from pipeline.agents.staleness import detect_stale

conn = get_db()
init_db(conn)
seed_db(conn)
print("DB initialized with sample providers.")

DB initialized with sample providers.


## Step 2: Detect Stale Records

In [2]:
stale = detect_stale(conn, days=90)
print(f"Found {len(stale)} stale records:\n")
for r in stale:
    print(f"  {r['provider_id']} | {r['provider_name']} | last verified: {r['last_verified_date']}")

Found 5 stale records:

  HL_001 | John Smith, MD | last verified: 2023-09-01
  HL_002 | Maria Garcia, DO | last verified: 2024-01-15
  HL_003 | Robert Lee, MD | last verified: 2022-06-10
  HL_004 | Susan Chen, MD | last verified: 2023-03-20
  HL_005 | James Wilson, MD | last verified: 2021-11-05


## Step 3: Run Pipeline on One Record (Mocked for Offline Demo)

In [3]:
from unittest.mock import patch

record = stale[0]
print(f"Processing: {record['provider_id']} — {record['provider_name']}\n")

mock_nppes = {
    "provider_name": record["provider_name"],
    "address": "250 Health Park Dr, Fort Myers, FL 33908",
    "phone": "239-555-9000",
    "specialty": record["specialty"],
    "active": True,
    "practice_name": "",
}
mock_cms = {
    "provider_name": record["provider_name"],
    "specialty": record["specialty"].upper(),
    "practice_name": "Fort Myers Medical Group",
    "active": True,
    "address": "250 Health Park Dr, Fort Myers, FL 33908",
    "phone": "239-555-9000",
}

with patch("pipeline.agents.fetch.fetch_nppes", return_value=mock_nppes), \
     patch("pipeline.agents.fetch.fetch_cms", return_value=mock_cms), \
     patch("pipeline.agents.fetch.fetch_florida_board", return_value=None), \
     patch("pipeline.agents.fetch.fetch_website", return_value=None):
    result = run_pipeline(record, conn)

print(f"Action: {result['recommended_action']}")
print(f"Confidence: {result['overall_confidence']}")
print(f"Reason: {result['reason']}\n")
for diff in result["diffs"]:
    print(f"  {diff['field']}: {diff['old_value']!r} → {diff['new_value']!r} "
          f"(score={diff['confidence_score']}, sources={diff['supporting_sources']})")

Processing: HL_001 — John Smith, MD

Action: auto_update
Confidence: 1.0
Reason: Updated fields confirmed by multiple reliable sources (overall confidence: 1.0).

  provider_name: 'John Smith, MD' → 'Smith John MD' (score=1.0, sources=['nppes', 'cms'])
  practice_name: 'ABC Heart Group' → 'Fort Myers Medical Group' (score=1.0, sources=['cms'])
  address: '100 Main St, Naples, FL 34102' → '250 HEALTH PARK DR, FORT MYERS, FL 33908' (score=1.0, sources=['nppes', 'cms'])
  phone: '239-555-1234' → '+12395559000' (score=1.0, sources=['nppes', 'cms'])


## Step 4: View Audit Log

In [4]:
import sqlite3
rows = conn.execute(
    "SELECT provider_id, field, old_value, new_value, confidence_score, action, reason "
    "FROM audit_log ORDER BY id DESC LIMIT 10"
).fetchall()
for row in rows:
    print(dict(row))

{'provider_id': 'HL_001', 'field': 'phone', 'old_value': '239-555-1234', 'new_value': '+12395559000', 'confidence_score': 1.0, 'action': 'auto_update', 'reason': 'Updated fields confirmed by multiple reliable sources (overall confidence: 1.0).'}
{'provider_id': 'HL_001', 'field': 'address', 'old_value': '100 Main St, Naples, FL 34102', 'new_value': '250 HEALTH PARK DR, FORT MYERS, FL 33908', 'confidence_score': 1.0, 'action': 'auto_update', 'reason': 'Updated fields confirmed by multiple reliable sources (overall confidence: 1.0).'}
{'provider_id': 'HL_001', 'field': 'practice_name', 'old_value': 'ABC Heart Group', 'new_value': 'Fort Myers Medical Group', 'confidence_score': 1.0, 'action': 'auto_update', 'reason': 'Updated fields confirmed by multiple reliable sources (overall confidence: 1.0).'}
{'provider_id': 'HL_001', 'field': 'provider_name', 'old_value': 'John Smith, MD', 'new_value': 'Smith John MD', 'confidence_score': 1.0, 'action': 'auto_update', 'reason': 'Updated fields con

## Cleanup

In [5]:
conn.close()
print("Demo complete.")

Demo complete.
